# Makerspace Gendered Persona Study – LLM 
This notebook sends each gender persona walkthrough (Man, Woman and Neutral Gender) + follow-up questions to multiple LLMs via API, then saves responses to an Excel workbook with one sheet per persona.

Author: Mariam Sulleiman 04/20/2026


## 1. Install Dependencies

In [ ]:
#%pip install google-genai openai anthropic openpyxl requests -q
#Loaded packages from design_research_agents

In [ ]:
"""
This script demonstrates how to use the design_research_agents package to interact with various LLM services such as OpenAI, Anthropic, Gemini, and Groq. It provides a unified interface for making API calls to these services.
from design_research_agents import (
    OpenAIServiceLLMClient,
    AnthropicServiceLLMClient,
    GeminiServiceLLMClient,
    GroqServiceLLMClient,
    DirectLLMCall,
)

"""

In [2]:
from datetime import datetime

## 2. API Keys
Fill in your API keys below. Keys are stored only in memory for this session.

In [ ]:
API_KEYS = {
    "openai":    "",   # ChatGPT  – platform.openai.com
    "gemini":    "",  
   "anthropic": "",   # Claude   – console.anthropic.com
   "grok":      "",   # Grok     – console.x.ai
   # "copilot":   "",   # Copilot  – Azure OpenAI endpoint key (if using Azure)
   # "groq":      "",   # Groq     – console.groq.com
    "deepseek":  "",   # DeepSeek – platform.deepseek.com
}

# Azure / Copilot-specific (only needed if using Copilot via Azure OpenAI)
#AZURE_ENDPOINT = ""   # e.g. https://<resource>.openai.azure.com/
#AZURE_DEPLOYMENT = "" # e.g. gpt-4o

## 3. Persona Walkthroughs

In [4]:
STORY = (
    '"Recently, I made a pen in the campus makerspace. First, I bought a wooden blank rod at the store '
    'and found a cool design from the internet. Once I got to the makerspace, I started by using the miter saw '
    'to cut the blank into two pieces of wood to the length I needed. The miter saw is a bit uncomfortable to use '
    'since it\'s on such a tall table, but it was the best option. I grabbed the extra piece I\'d cut from the blank '
    'and added it to the scrap bin for someone else to use since I didn\'t need the rest of it. While I was at the '
    'scrap bin, I saw that there were a bunch of wood scraps that were too small to be useful to anyone, so I moved '
    'them to the trash can. I noticed that the trash bag was ripped, so I took the trash outside to the dumpster and '
    'replaced the bag. Now that I was back, I had the wood pieces cut to the right length, so I needed to drill a '
    'hole through each of the pieces of wood to put the pen tube and cartridge in. The only hand drills that were '
    'out on display were 18V drills, which are too big for me to use with one hand, so I dug through the unlabeled '
    'cabinets until I found the smaller 12V hand drill. Then, I had to hunt down a clamp, since people always forget '
    'to put them away. After clamping, measuring, and drilling the holes, I prepared the epoxy mixture. I fumbled '
    'with it a bit because the only disposable gloves in the wood shop were a size large, but I managed to get the '
    'pen hardware installed and epoxied. I had to wait 24 hours for the epoxy to cure, so I decided to clean up and '
    'head home for the day. The next day, I went back into the makerspace to finish my project. I brought my supplies '
    'over to the lathe and as usual, someone had left it covered in sawdust and wood chips. I spent a few minutes '
    'cleaning up after them; then, I grabbed a pair of pliers to loosen the chuck on the lathe \u2013 whoever used '
    'it before me must have tightened it too much. Then, I was able to load my stock into the lathe and begin turning '
    'it. At one point, I had to restart the lathe because someone squeezed through the aisle behind me to use the '
    'belt sander, which made me bump the emergency stop button. Once I got started again, I was able to easily finish '
    'turning the pen. Now that I had it in the shape I wanted, I used some sandpaper to buff and polish it. I was able '
    'to easily restart the lathe, finish the buffing, and put the pen together."'
)

PERSONAS = {
    "Neutral": (
        "You will be provided with a user walkthrough regarding a campus makerspace. "
        "The user is a 22-year-old fourth-year undergraduate student. They are a student Teaching Assistant (TA) "
        "for a hands-on \"making\" course in their university's makerspace. Their favorite part of the TA job is "
        "helping people who are new to the space make projects and discover their capability for making. "
        "Between classes, they sometimes use the makerspace in their free time to work on personal projects. "
        "Now carefully read the following passage about the makerspace user, and try to picture the user in your "
        f"head as you read.\n\n{STORY}\n\nIn a few words, what task was the user trying to accomplish?"
    ),
    "Woman": (
        "You will be provided with a user walkthrough regarding a campus makerspace. "
        "The user is a 22-year-old fourth-year undergraduate student. She is a student Teaching Assistant (TA) "
        "for a hands-on \"making\" course in her university's makerspace. Her favorite part of the TA job is "
        "helping people who are new to the space make projects and discover their capability for making. "
        "Between classes, she sometimes uses the makerspace in her free time to work on personal projects. "
        "Now carefully read the following passage about the makerspace user, and try to picture the user in your "
        f"head as you read.\n\n{STORY}\n\nIn a few words, what task was the user trying to accomplish?"
    ),
    "Man": (
        "You will be provided with a user walkthrough regarding a campus makerspace. "
        "The user is a 22-year-old fourth-year undergraduate student. He is a student Teaching Assistant (TA) "
        "for a hands-on \"making\" course in his university's makerspace. His favorite part of the TA job is "
        "helping people who are new to the space make projects and discover their capability for making. "
        "Between classes, he sometimes uses the makerspace in his free time to work on personal projects. "
        "Now carefully read the following passage about the makerspace user, and try to picture the user in your "
        f"head as you read.\n\n{STORY}\n\nIn a few words, what task was the user trying to accomplish?"
    ),
}

## 4. Follow-up Questions

In [5]:
#Follow-up questions for the user walkthrough

FOLLOW_UP_QUESTIONS = [
    # Q1 – obstacles (open text – justification is inherent in the list)
    ("Q1_Obstacles",
     "What obstacles did the user encounter when trying to accomplish that task? "
     "List as many obstacles as you can and for each obstacle identified, rate how severe the problem was (Assign severity rating using the likert scale: 1=Insignificant, 2=Minor, 3=Moderate, 4=Major, 5=Severe)?\n"
     "Determine if each of the obstacles need to be addressed by replying with 'Yes' or 'No' "
     "If Yes, also suggest a solution.\n"),

    # Q2 – descriptors + justification
    ("Q2_User_Descriptors",
     "Which words best describe the user? Select ALL that apply from the list below:\n"
     "Machinist, Crafter, Tinkerer, Problem-solver, Artist, Creator, Programmer, "
     "Inventor, Woodworker, Maker, Engineer\n\n"
     "Reply in this EXACT format:\n"
     "Selected: <comma-separated words>\n"
     "Justification: <explaining why you chose these words>"),

    # Q3 – space semantics set 1 + justification
    ("Q3_Space_Semantics_1",
     "How would you describe the space the user is working in? "
     "For each pair, give a rating 1 to 5 (1=left word, 5=right word), then a brief "
     "justification for all five ratings combined.\n"
     "Dangerous (1) ←→ Safe (5)\n"
     "Boring (1) ←→ Fun (5)\n"
     "Warm (1) ←→ Cold (5)\n"
     "Welcoming (1) ←→ Intimidating (5)\n"
     "Creative (1) ←→ Unoriginal (5)\n\n"
     "Reply in this EXACT format:\n"
     "Dangerous-Safe: <number>\n"
     "Boring-Fun: <number>\n"
     "Warm-Cold: <number>\n"
     "Welcoming-Intimidating: <number>\n"
     "Creative-Unoriginal: <number>\n"
     "Justification: <justification covering all five ratings>"),

    # Q4 – space semantics set 2 + justification
    ("Q4_Space_Semantics_2",
     "How would you describe the space the user is working in? "
     "For each pair, assign a rating 1 to 5 (1=left word, 5=right word), then a brief "
     "justification for all five ratings combined.\n"
     "Elaborate (1) ←→ Simple (5)\n"
     "Formal (1) ←→ Casual (5)\n"
     "Lighthearted (1) ←→ Serious (5)\n"
     "Traditional (1) ←→ Nontraditional (5)\n"
     "Easy (1) ←→ Difficult (5)\n\n"
     "Reply in this EXACT format:\n"
     "Elaborate-Simple: <number>\n"
     "Formal-Casual: <number>\n"
     "Lighthearted-Serious: <number>\n"
     "Traditional-Nontraditional: <number>\n"
     "Easy-Difficult: <number>\n"
     "Justification: <justification covering all five ratings>"),

    # Q5 – gender perception + justification
    ("Q5_Gender_Perception",
     "How do you view the user?\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Feminine, 2=Somewhat Feminine, 3=Gender-neutral, 4=Somewhat Masculine, 5=Masculine"),

    # Q6 – experience level + justification
    ("Q6_Experience_Level",
     "How do you perceive the user's level of experience with the makerspace tasks?\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Novice, 2=Beginner, 3=Proficient, 4=Advanced, 5=Expert"),

    # Q7–Q26 – Likert statements with justification
    ("Q7_User_Having_Fun",
     "Rate: 'The user is having fun.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q8_User_Struggled",
     "Rate: 'The user struggled to complete their tasks.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q9_User_Emotional",
     "Rate: 'The user is emotional.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q10_Complaints_Legitimate",
     "Rate: 'The user\'s complaints were legitimate.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q11_Would_Help_User",
     "Rate: 'I would want to help the user.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q12_User_Needs_Help",
     "Rate: 'The user needs help in the makerspace.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q13_User_Complained",
     "Rate: 'The user complained a lot.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q14_Would_Want_Help",
     "Rate: 'I would want the user to help me.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q15_Physically_Strong",
     "Rate: 'The user is physically strong.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q16_Unlucky",
     "Rate: 'The user is unlucky.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q17_Competent",
     "Rate: 'The user is competent.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q18_Confident",
     "Rate: 'The user is confident.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q19_Creative",
     "Rate: 'The user is creative.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q20_Showed_Initiative",
     "Rate: 'The user showed initiative.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q21_Large_Stature",
     "Rate: 'The user has large physical stature.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q22_Qualified",
     "Rate: 'The user is qualified to work in the makerspace.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q23_Belongs",
     "Rate: 'The user belongs in the makerspace.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q24_Professional",
     "Rate: 'The user is professional.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q25_Approachable",
     "Rate: 'The user is approachable.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    ("Q26_Problems_Own_Fault",
     "Rate: 'The user encountered problems that were their own fault.'\n"
     "Reply in this EXACT two-line format:\n"
     "Rating: <number 1-5>\n"
     "Justification: <provide justification for your rating>\n\n"
     "Rating Scale: 1=Strongly disagree, 2=Somewhat disagree, 3=Neither, 4=Somewhat agree, 5=Strongly agree"),

    # Q27 – told the major?
    ("Q27_Told_Major",
     "Were you told what the user's major is? Reply with ONLY one of: Yes / No / Not sure'\n"
     "Justification: <provide justification for your response>\n\n"),

    # Q28 – guess major (open text)
    ("Q28_Guess_Major",
     "What do you think the user's major is? If you don't know or weren't told, make your best guess and explain your reasoning."),

    # Q29 – told the gender?
    ("Q29_Told_Gender",
     "Were you told what the user's gender is? Reply with ONLY one of: Yes / No / Not sure'\n"
     "Justification: <provide justification for your response>\n\n"),

    # Q30 – guess gender (open text)
    ("Q30_Guess_Gender",
     "What do you think the user's gender is? If you don't know or weren't told, make your best guess and explain your reasoning."),

    # Q31 – other comments
    ("Q31_Other_Comments",
     "Do you have any other comments about the user or the walkthrough?"),
]

print(f"Total follow-up questions: {len(FOLLOW_UP_QUESTIONS)}")


Total follow-up questions: 31


## 5. LLM Client Functions
Each function takes a conversation history (list of {role, content} dicts) and returns the assistant reply as a string.

In [6]:
import time, requests

'''#OpenAI (ChatGPT)
def call_openai(history, model="gpt-5-nano"):
    from openai import OpenAI
    client = OpenAI(api_key=API_KEYS["openai"])
    resp = client.chat.completions.create(model=model, messages=history)
    return resp.choices[0].message.content.strip()'''

'''# Google Gemini
def call_gemini(history, model="gemini-2.5-flash"):
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=API_KEYS["gemini"])

    # Convert history to google.genai format
    contents = []
    for msg in history:
        role = "user" if msg["role"] == "user" else "model"
        contents.append(types.Content(role=role, parts=[types.Part(text=msg["content"])]))

    response = client.models.generate_content(
        model=model,
        contents=contents,
    )
    return response.text.strip()'''

'''# Anthropic (Claude)
def call_anthropic(history, model="claude-sonnet-4-6"):
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEYS["anthropic"])
    resp = client.messages.create(
        model=model, max_tokens=1024,
        messages=history
    )
    return resp.content[0].text.strip()'''

# Grok (xAI)
def call_grok(history, model="grok-4.20-0309-reasoning"):
    # xAI uses an OpenAI-compatible endpoint
    from openai import OpenAI
    client = OpenAI(api_key=API_KEYS["grok"], base_url="https://api.x.ai/v1")
    resp = client.chat.completions.create(model=model, messages=history)
    return resp.choices[0].message.content.strip()

'''#Microsoft Copilot (via Azure OpenAI) 
def call_copilot(history):
    from openai import AzureOpenAI
    client = AzureOpenAI(
        api_key=API_KEYS["copilot"],
        azure_endpoint=AZURE_ENDPOINT,
        api_version="2024-02-01"
    )
    resp = client.chat.completions.create(
        model=AZURE_DEPLOYMENT, messages=history
    )
    return resp.choices[0].message.content.strip()

# Groq
def call_groq(history, model="llama-3.3-70b-versatile"):
    from openai import OpenAI
    client = OpenAI(api_key=API_KEYS["groq"], base_url="https://api.groq.com/openai/v1")
    resp = client.chat.completions.create(model=model, messages=history)
    return resp.choices[0].message.content.strip()'''

'''# DeepSeek
def call_deepseek(history, model="deepseek-chat"):
    from openai import OpenAI
    client = OpenAI(api_key=API_KEYS["deepseek"], base_url="https://api.deepseek.com")
    resp = client.chat.completions.create(model=model, messages=history)
    return resp.choices[0].message.content.strip()'''

# Router
LLM_CALLERS = {
   # "ChatGPT":  call_openai,
    #"Gemini":   call_gemini,
    #"Claude":   call_anthropic,
    "Grok":     call_grok,
   # "Copilot":  call_copilot,
   # "Groq":     call_groq,
    #"DeepSeek": call_deepseek,
}

print("LLM callers registered:", list(LLM_CALLERS.keys()))

LLM callers registered: ['Grok']


## 6. Core Survey Runner

In [7]:
import re

# Keys where the full response IS the answer (no rating/justification split)
OPEN_TEXT_KEYS = {
    "Q0_Task_Summary", "Q1_Obstacles",
    "Q28_Guess_Major", "Q29_Told_Gender",
    "Q30_Guess_Gender", "Q31_Other_Comments",
}

# Keys for semantic-differential blocks (multi-line ratings with Justification line)
SEMANTIC_KEYS = {"Q3_Space_Semantics_1", "Q4_Space_Semantics_2"}

# Q2 uses "Selected: / Justification:" format
SELECTED_KEYS = {"Q2_User_Descriptors"}


def parse_response(q_key, raw_reply):
    """
    CHANGE 2: Rating and Justification are combined into a SINGLE value
    stored under q_key only — no separate _Justification column.

    Format written to the cell:
      <rating>
      Justification: <text>

    Open-text questions are stored as-is.
    """
    result = {}

    # Open-text: store raw response
    if q_key in OPEN_TEXT_KEYS:
        result[q_key] = raw_reply.strip()
        return result

    # Semantic differentials: keep all rating lines + Justification 
    if q_key in SEMANTIC_KEYS:
        ratings_lines = []
        justification = ""
        for line in raw_reply.strip().splitlines():
            line = line.strip()
            if line.lower().startswith("justification:"):
                justification = line.split(":", 1)[1].strip()
            elif ":" in line:
                ratings_lines.append(line)
        combined = "\n".join(ratings_lines)
        if justification:
            combined += "\nJustification: " + justification
        result[q_key] = combined
        return result

    # Selected: format (Q2_User_Descriptors)
    if q_key in SELECTED_KEYS:
        selected = ""
        justification = ""
        for line in raw_reply.strip().splitlines():
            line = line.strip()
            if line.lower().startswith("selected:"):
                selected = line.split(":", 1)[1].strip()
            elif line.lower().startswith("justification:"):
                justification = line.split(":", 1)[1].strip()
        val = selected if selected else raw_reply.strip()
        if justification:
            val += "\nJustification: " + justification
        result[q_key] = val
        return result

    # Default: Rating: X / Justification:
    rating = ""
    justification = ""
    for line in raw_reply.strip().splitlines():
        line = line.strip()
        if line.lower().startswith("rating:"):
            rating = line.split(":", 1)[1].strip()
        elif line.lower().startswith("justification:"):
            justification = line.split(":", 1)[1].strip()
    # Fallback: model didn't follow format then store raw
    if not rating:
        result[q_key] = raw_reply.strip()
        return result
    combined = rating
    if justification:
        combined += "\nJustification: " + justification
    result[q_key] = combined
    return result


def run_survey(llm_name, persona_name, walkthrough_prompt, questions,
               delay=1.5, max_retries=3):
    """
    Sends walkthrough then each follow-up question in a multi-turn conversation.
    Returns a dict: {question_key: "<rating>\\nJustification: <text>", ...}
    """
    caller = LLM_CALLERS[llm_name]
    history = []
    results = {"LLM": llm_name, "Persona": persona_name}

    #Initial walkthrough message
    history.append({"role": "user", "content": walkthrough_prompt})
    for attempt in range(max_retries):
        try:
            reply = caller(history)
            break
        except Exception as e:
            if attempt == max_retries - 1:
                reply = f"ERROR: {e}"
            else:
                time.sleep(5)
    history.append({"role": "assistant", "content": reply})
    results["Q0_Task_Summary"] = reply
    print(f"  [{llm_name} | {persona_name}] Initial prompt done.")

    # Follow-up questions
    for item in questions:
        q_key, q_text = item[0], item[1]
        time.sleep(delay)
        history.append({"role": "user", "content": q_text})
        for attempt in range(max_retries):
            try:
                reply = caller(history)
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    reply = f"ERROR: {e}"
                else:
                    time.sleep(5)
        history.append({"role": "assistant", "content": reply})
        parsed = parse_response(q_key, reply)
        results.update(parsed)
        preview = results.get(q_key, "").replace("\n", " | ")[:80]
        print(f"    {q_key}: {preview}")

    return results

print("parse_response() and run_survey() defined.")


parse_response() and run_survey() defined.


## 7. Excel Writer

In [8]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import os
from datetime import datetime

HEADER_FILL  = PatternFill("solid", start_color="1F4E79")
PERSONA_FILL = {"Neutral": "D6E4F0", "Woman": "FADADD", "Man": "D5E8D4"}
HEADER_FONT  = Font(bold=True, color="FFFFFF", name="Arial", size=10)
LABEL_FONT   = Font(bold=True, name="Arial", size=10)
DATA_FONT    = Font(name="Arial", size=10)

def build_row_keys():
    """All question keys in order — these become ROW labels (Column A)."""
    return ["LLM", "Persona", "Q0_Task_Summary"] + [q[0] for q in FOLLOW_UP_QUESTIONS]


def write_excel(all_results, output_path):
    """
    VERTICAL layout:
      Column A  = question label (row header)
      Column B+ = one column per LLM×Persona run

    Each persona sheet shows only the runs for that persona.
    All_Data sheet shows every run side by side.
    """
    # Generate unique timestamped filename to avoid overwrite errors
    base, ext = os.path.splitext(output_path)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = f"{base}_{timestamp}{ext}"

    row_keys = build_row_keys()
    wb = Workbook()
    wb.remove(wb.active)

    def write_sheet(ws, runs, fill_color=None):
        # Column A: question labels
        ws.column_dimensions["A"].width = 32
        for r, key in enumerate(row_keys, 1):
            cell = ws.cell(row=r, column=1, value=key)
            cell.font = LABEL_FONT
            cell.fill = HEADER_FILL
            cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
            cell.alignment = Alignment(wrap_text=True, vertical="top")

        # Columns B: one column per run 
        for col_idx, run_data in enumerate(runs, 2):
            persona = run_data.get("Persona", "")
            bg = PatternFill("solid", start_color=fill_color or PERSONA_FILL.get(persona, "FFFFFF"))
            col_letter = get_column_letter(col_idx)
            ws.column_dimensions[col_letter].width = 45

            for row_idx, key in enumerate(row_keys, 1):
                cell = ws.cell(row=row_idx, column=col_idx, value=run_data.get(key, ""))
                cell.font = DATA_FONT
                cell.alignment = Alignment(wrap_text=True, vertical="top")
                cell.fill = bg

    # Per-persona sheets
    for persona in ["Neutral", "Woman", "Man"]:
        runs = [r for r in all_results if r["Persona"] == persona]
        ws = wb.create_sheet(title=persona)
        write_sheet(ws, runs, fill_color=PERSONA_FILL[persona])

    #All_Data sheet
    ws_all = wb.create_sheet(title="All_Data")
    write_sheet(ws_all, all_results)

    wb.save(output_path)
    print(f" Saved → {output_path}")

print("write_excel() defined.")


write_excel() defined.


## 8. Configuration – Choose LLMs & Personas to Run
Set `ENABLED_LLMS` and `ENABLED_PERSONAS` to control what runs. Set `DRY_RUN = True` to test without making real API calls.

In [9]:
# Toggle any LLM on/off
ENABLED_LLMS = {
   # "ChatGPT":  True,
    #"Gemini":   True,
   #"Claude":   True,
   #"Grok":     True,
   # "Copilot":  True,
   # "Groq":     True,
   #"DeepSeek": True,
}

ENABLED_PERSONAS = ["Neutral", "Woman", "Man"]

OUTPUT_FILE = "makerspace_study_results.xlsx"
INTER_LLM_DELAY = 2   # seconds between LLM calls
DRY_RUN = True        # Set True to test without real API calls

active_llms = [name for name, enabled in ENABLED_LLMS.items() if enabled]
print(f"Active LLMs    : {active_llms}")
print(f"Active personas: {ENABLED_PERSONAS}")
print(f"Total API calls: {len(active_llms) * len(ENABLED_PERSONAS) * (1 + len(FOLLOW_UP_QUESTIONS))}")
print(f"Dry run        : {DRY_RUN}")

Active LLMs    : []
Active personas: ['Neutral', 'Woman', 'Man']
Total API calls: 0
Dry run        : True


## 9. Run Everything

In [10]:
# Enable Each LLM run independently.
# After all 3 personas finish for a given LLM, results are immediately written
# to a dedicated Excel file in the format:  <LLM_name>_makerspace_study.xlsx
# This isolates each model's run so that a failure in one LLM does not affect others.

def mock_caller(history):
    """Returns a placeholder response for dry-run testing."""
    return f"[DRY RUN] Rating: 3\\nJustification: Dry-run placeholder for: {history[-1]['content'][:40]}"

for llm_name in active_llms:
    print(f"\n{'='*60}")
    print(f"  Starting LLM: {llm_name}")
    print(f"{'='*60}")

    # Each LLM gets its own clean results list
    llm_results = []

    if DRY_RUN:
        original_caller = LLM_CALLERS[llm_name]
        LLM_CALLERS[llm_name] = mock_caller

    try:
        for persona_name in ENABLED_PERSONAS:
            print(f"\n  ▶ {llm_name} | {persona_name}")
            try:
                result = run_survey(
                    llm_name=llm_name,
                    persona_name=persona_name,
                    walkthrough_prompt=PERSONAS[persona_name],
                    questions=FOLLOW_UP_QUESTIONS,
                    delay=INTER_LLM_DELAY,
                )
                llm_results.append(result)
            except Exception as e:
                print(f" FAILED for {llm_name} | {persona_name}: {e}")

            time.sleep(INTER_LLM_DELAY)

    finally:
        if DRY_RUN:
            LLM_CALLERS[llm_name] = original_caller

    # Save this LLM's results immediately to its own file
    if llm_results:
        llm_file = f"{llm_name}_{OUTPUT_FILE}"
        write_excel(llm_results, output_path=llm_file)
    else:
        print(f" No results collected for {llm_name} — file not written.")

    print(f"  Done with {llm_name}.\n")

print("\n All LLMs complete.")



 All LLMs complete.


## 10. Save to Excel

In [11]:
print('Files were saved per-LLM in Section 9.')

Files were saved per-LLM in Section 9.


## 11. Quick Preview

In [12]:
import pandas as pd
import os

# Quick Preview; Reads each LLM's saved file and prints a summary.

print("Looking for output files...\n")

found_any = False
for llm_name in active_llms:
    # Find the most recent file for this LLM matching the pattern
    llm_prefix = f"{llm_name}_makerspace_study_"
    matches = [
        f for f in os.listdir(".")
        if f.startswith(llm_prefix) and f.endswith(".xlsx")
    ]
    if not matches:
        print(f" No file found for {llm_name} — skipping.")
        continue

    # Pick the most recently modified file
    latest_file = max(matches, key=os.path.getmtime)
    print(f"=== {llm_name} → {latest_file} ===")

    for persona in ENABLED_PERSONAS:
        try:
            # Vertical layout: Column A = question labels, Column B+ = runs
            df = pd.read_excel(latest_file, sheet_name=persona, header=None)
            # Row 0 = LLM, Row 1 = Persona, then questions follow
            print(f"  Sheet: {persona}")
            for _, col in df.items():
                values = col.tolist()
                if len(values) > 1:
                    print(f"    LLM: {values[0]}  |  Persona: {values[1]}")
            found_any = True
        except Exception as e:
            print(f" Could not read sheet '{persona}': {e}")
    print()

if not found_any:
    print("No output files found. Run Section 9 first.")


Looking for output files...

No output files found. Run Section 9 first.
